# 13 · 收尾：运维、成本，和怎么讲这个项目

## 这一章在整个项目里的位置

```
00–12 ✅ → [13 收尾]
            告警 · 日志 · 成本 · 架构图 · 讲稿 · 简历   ← 你在这里
```

## 背景：为什么需要这一章

到 12 号为止，平台的每一层都搭好了：数据进来、训练、考试、2.18 亿个分数、API、知识库问答、Agent。但一个系统上线以后，工程师的活还没完：
- **它现在健康吗？** 出错、变慢，要在用户发现之前知道 → **告警**
- **谁在用、用得怎么样？** → **日志**
- **花了多少钱？** → **成本**
- **你能不能在 2 分钟里把它讲清楚？** → **架构图、讲稿、简历**

前三件是"运维"，公司里叫 operations 或 SRE 的一部分；最后一件是你找工作要用的。

## 你在哪里操作

```
你：在这个 notebook 里运行一格（约 10 分钟）
看结果：AWS 控制台 → CloudWatch → Alarms（告警）和 Dashboards → qafi-dev（仪表盘）
```

## 这一章做的 6 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 告警和仪表盘 | 给系统装上烟雾报警器和一块仪表盘 |
| 2 | 看日志 | 翻看出入记录：谁来过、办成了没有 |
| 3 | 成本 | 看账单 |
| 4 | 架构图 | 一张图讲清全貌 |
| 5 | 讲稿 | 30 秒、2 分钟的项目介绍，和常见追问 |
| 6 | 简历条目 | 把做过的事写成招聘方看得懂的几行 |

**花费**：几乎为 0（5 个告警每月约 0.5 美元；仪表盘在免费额度内）。

**开始前**：终端运行 `aws login --profile qafi`。

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import json, os, subprocess, time, warnings
from datetime import date
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import boto3
import pandas as pd
from qafi_platform import operations
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"
pd.set_option("display.max_colwidth", 120)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
def terraform_output(name):
    return json.loads(subprocess.run(f"terraform -chdir=infra/dev output -json {name}", shell=True, capture_output=True, text=True).stdout)
aws = boto3.Session(region_name="eu-central-1")
print("AWS:", aws.client("sts").get_caller_identity()["Arn"])

---

## 第 1 步 · 告警和仪表盘

### 在干什么
给平台装上告警：出问题时自动通知你；再做一块仪表盘，一页看清平台在干什么。

### 背景
比喻：**烟雾报警器和仪表盘**。你不会每天盯着厨房看有没有着火，而是装一个报警器，冒烟了它叫你。

**报警器装在哪里？** 只装在**用户或账单会先发现问题**的地方：

| 告警 | 什么时候响 | 为什么要它 |
|---|---|---|
| `qafi-dev-api-5xx` | API 回答"服务器错误"（5xx） | 用户拿不到答案了 |
| `qafi-dev-api-slow` | 最慢的 5% 请求连续 15 分钟超过 5 秒 | 用户等太久 |
| `qafi-dev-lambda-errors` | API 函数报错 | 代码或权限出问题 |
| `qafi-dev-lambda-throttles` | 请求因为并发太多被拒 | 容量不够 |
| `qafi-dev-athena-bytes` | Athena 一天扫描超过 10 GB | 有人（或某个程序）查得太多，要花钱了 |

整个账单还有 00 号设的**预算告警**（10、25、50 美元三档），这里不重复。

**告警的三种状态**：`OK`（正常）、`ALARM`（响了）、`INSUFFICIENT_DATA`（还没有数据）。没有请求时不算出问题（`treat_missing_data = notBreaching`），否则一个没人用的 API 会一直报警。

**通知**：告警响了，就往一个 **SNS 主题**（`qafi-dev-alerts`）发消息，订阅了这个主题的邮箱会收到邮件。邮箱地址不写进 Git，运行时作为参数传给 Terraform；AWS 会先发一封确认邮件，点了确认才会开始收告警。

**仪表盘** `qafi-dev`：一页上看 API 的请求数和错误数、延迟（中位数和最慢的 5%）、Lambda 的调用和错误、Athena 每小时扫了多少、Claude 每小时用了多少词元（这一块在爱尔兰）。

### 用了什么技术，为什么
- **CloudWatch**：AWS 的监控服务。各个服务自动把指标发给它（请求数、错误数、延迟……），告警和仪表盘都建在这些指标上
- **SNS**：AWS 的消息通知服务，一个主题可以接邮件、短信、Slack 等
- 都写在 Terraform 里（`infra/dev/monitoring.tf`），和别的资源一样审核、一样能重建

### 执行
想收邮件就在下面填上你的邮箱（不想收就留空，告警照样在控制台里看得到）：

In [ ]:
ALERT_EMAIL = ""   # 例如 "you@example.org"；留空就不发邮件
email = f"-var alert_email={ALERT_EMAIL}" if ALERT_EMAIL else ""
run(f"set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color {email} -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")
operations.alarms(aws.client("cloudwatch"), "qafi-dev")

### 结果怎么看
- 第一次运行：`Plan: 7 to add`（一个通知主题、5 个告警、一个仪表盘）；填了邮箱会多一个订阅。已经建好的话是 `No changes`
- 下面的表：5 个告警和它们的状态。刚建好时是 `INSUFFICIENT_DATA`，几分钟后会变成 `OK`
- 填了邮箱：去邮箱里点 **Confirm subscription**

### 去网页上看
AWS 控制台 → **CloudWatch** → **Dashboards** → `qafi-dev`。之前几章的调用都会出现在图上。

### 公司里什么情况怎么做
- 告警要少而准：只在"需要有人去处理"的时候响。响得太多，大家就不看了（"告警疲劳"）
- 每个告警都要写清楚**它说明了什么**（我们写在 `alarm_description` 里），半夜被叫醒的人才知道从哪里查起
- 大公司会把告警接到值班系统（PagerDuty、Opsgenie），我们接邮件就够了

### 面试怎么说
> "CloudWatch alarms watch what users and the bill would notice first: 5xx errors, p95 latency, Lambda errors and throttles, and Athena bytes per day, all in Terraform and routed through SNS. One dashboard shows traffic, latency, errors, Athena scans and Claude token use."

---

## 第 2 步 · 看日志

### 在干什么
用 SQL 一样的语句查 API 的日志：多少请求、成功率、快慢，大家在查什么。

### 背景
比喻：**翻看出入记录**。10 号的 API 每处理一个请求，就往日志里写一行 JSON：

```
{"path": "/v1/proteins/TP53/variants/R175H", "status": 200, "ms": 1}
```

这种有结构的日志（**structured logging**）可以直接统计。**CloudWatch Logs Insights** 是查日志的工具，语法像 SQL：`filter` 筛选，`stats` 统计，`parse` 从文字里切出字段。

### 执行

In [ ]:
logs = aws.client("logs")
health = operations.logs(logs, "/aws/lambda/qafi-dev-api", """
    fields path, status, ms | filter ispresent(path) and path != "/health"
    | stats count(*) as requests, pct(ms, 50) as p50_ms, pct(ms, 95) as p95_ms by status""", hours=72)
popular = operations.logs(logs, "/aws/lambda/qafi-dev-api", """
    fields path | filter ispresent(path) and path like /proteins/
    | parse path "/v1/proteins/*/*" as protein, rest
    | stats count(*) as requests by protein | sort requests desc | limit 5""", hours=72)
display(health)
popular

### 结果怎么看
第一张表按回答状态统计最近三天的请求（写这一章时）：
- `200`（成功）：48 次，中位数 **1 毫秒**（答案在函数的内存里），最慢的 5% 约 **3 秒**（要查 Athena 的那些）
- `422`（格式不对）：6 次；`404`（没有这个蛋白）：1 次。都是我们故意测的错误输入

`ms` 只是函数里面花的时间，不包括网络；用户感受到的时间在 API Gateway 的访问日志和仪表盘的延迟图里。

第二张表是被查得最多的蛋白：TP53 最多。真实上线后，这张表告诉你用户关心什么。

**漂移**（drift）监控在这里怎么做？我们的分数是一版一版批量算好的，模型不会在线上悄悄变。所以要盯的是两件事：
1. **用户在问什么**：上面这张表。如果大家开始查我们从来没评估过的蛋白类型，要知道
2. **新一版和上一版差多少**：09c 第 8 步的新旧对比。下一版上线前也要做

### 面试怎么说
> "The API writes one JSON line per request, so Logs Insights gives request counts, status mix and latency percentiles directly. For a batch-scored product, drift monitoring means watching the query mix and comparing each release with the previous one before publishing it."

---

## 第 3 步 · 成本

### 在干什么
看这个月平台在 AWS 上花了多少钱，花在哪里，以及以后每个月大概要多少。

### 背景
比喻：**看账单**。免费计划的赠送额度会抵掉账单，所以账单上看起来是 0。我们看的是**抵扣之前**的金额：这才是平台真实的花费，也是换成付费账号以后要付的钱。

### 执行

In [ ]:
today = date.today()
bill = operations.costs(boto3.Session(region_name="us-east-1").client("ce"), today.replace(day=1).isoformat(), today.isoformat())
print(f"{today:%Y-%m} 到今天，抵扣额度之前一共 {bill['usd'].sum():.2f} 美元")
bill

### 结果怎么看
写这一章时（10 月 1–15 日）：**一共 2.16 美元**，全部由赠送额度抵掉。

| 服务 | 美元 | 花在哪里 |
|---|---|---|
| S3 | 1.44 | 存储：raw 里的原始数据（AlphaFold、旧预测压缩包……），湖里的 2.6 GB |
| Bedrock | 0.38 | Titan 把 2 万篇文档变成向量，Claude 回答问题 |
| KMS | 0.24 | 平台的加密密钥（每月 1 美元，按天计） |
| Secrets Manager | 0.08 | Airflow 的 Databricks 密钥（每月 0.4 美元） |
| Athena | 0.02 | 所有的查询 |

**每次使用花多少**（前面各章量出来的）：

| 一次…… | 大约 |
|---|---|
| API 查一个突变（Athena 最低收费 + Lambda） | 0.005 美分 |
| 知识库问答一次（Claude） | 0.3 美分 |
| Agent 回答一个问题 | 0.4–0.6 美分 |
| 全蛋白组导出一版到 S3（Databricks 免费版） | 0 |

**放着不用，每月大约 3–5 美元**：主要是 S3 存储，加上密钥 1 美元、Secrets Manager 0.4 美元、告警 0.5 美元。Lambda、API Gateway、Athena、S3 Vectors 都是按次付费，不用不花钱。这就是第 1 号决定"只用按量计费的服务"的效果。

**想省到几乎为 0**：`terraform destroy` 可以删掉计算和服务部分，数据桶保留，需要时再 `apply` 重建。找工作期间建议**保留**：面试时能现场演示一个真的在跑的 API。

### 公司里什么情况怎么做
- 每个资源打上标签（我们的 `project`、`environment`），按标签看成本
- **单位成本**比总额更有用："每千次查询多少钱"能直接拿来做容量规划和定价
- 用完就删的东西（开发环境、实验集群）要有到期机制

### 面试怎么说
> "The platform costs about two dollars for the first half of the month before credits, mostly S3 storage; everything that serves traffic is pay-per-request, so idle cost is a few dollars a month. A lookup costs about 0.005 cents, a cited RAG answer about 0.3 cents, an agent answer about half a cent."

---

## 第 4 步 · 架构图

### 在干什么
用一张图讲清整个平台：数据从哪里来、在哪里加工、模型在哪里训练和打分、结果怎么交给用户。

### 背景
面试里"讲讲你的项目"，第一件事就是画这张图。下面的图在 VS Code 和 GitHub 里都会直接显示成图（Mermaid 格式，代码就是图）：

```mermaid
flowchart LR
  subgraph Sources[公开数据源]
    U[UniProt]
    C[ClinVar]
    P[ProteinGym / MaveDB]
    A[AlphaFold]
    E[ESM-2 weights]
  end
  subgraph AWS_FRA[AWS 法兰克福]
    RAW[(S3 raw)]
    LAMBDA_IN[Lambda: fetch + validate]
    LAKE[(S3 lake<br/>Parquet, 2.18亿行)]
    GLUE[Glue Catalog + Athena]
    API[API Gateway + Lambda<br/>FastAPI 容器]
    CW[CloudWatch 告警 / 仪表盘]
  end
  subgraph DBX[Databricks]
    BRONZE[bronze] --> SILVER[silver] --> GOLD[gold<br/>dbt]
    MLF[MLflow 实验<br/>+ Unity Catalog 模型注册]
  end
  subgraph MAC[笔记本 GPU]
    TRAIN[训练 / 调参<br/>PyTorch, XGBoost, Optuna]
    SCORE[全蛋白组打分<br/>ESM-2 650M]
  end
  subgraph AWS_IRL[AWS 爱尔兰]
    KB[Bedrock 知识库<br/>Titan + S3 Vectors]
    CLAUDE[Claude Haiku 4.5]
    GR[Guardrails]
    AGENT[Agent 循环]
  end
  Sources --> LAMBDA_IN --> RAW --> BRONZE
  GOLD --> TRAIN --> MLF
  MLF --> SCORE --> GOLD
  GOLD -->|导出| LAKE --> GLUE --> API
  U --> KB
  AGENT --> API
  AGENT --> KB
  AGENT --> CLAUDE
  GR --- AGENT
  API --- CW
```

**怎么讲这张图**（从左到右，每一段一句话）：
1. **接入**：公开数据由 Lambda 从官方地址直接搬进 S3 raw，每个文件校验、登记，版本写死（03 号）
2. **湖仓**：Databricks 把原始文件变成 bronze → silver → gold 三层表，dbt 管 gold 层的模型和测试，Lakeflow Jobs 和 Airflow 编排（04–06 号）
3. **机器学习**：特征来自 ESM-2 语言模型、AlphaFold 结构和理化性质；按蛋白相似度分组做交叉验证，测试蛋白锁住只开一次；MLflow 记录每次实验，过了门禁的模型进注册表（07–09 号）
4. **批量打分**：2.18 亿个突变一次算好，结果回到平台（09c）
5. **服务**：分数导出到 S3 数据湖，Athena 查，FastAPI 容器跑在 Lambda 上，API Gateway 只放行签名请求（10 号）
6. **生成式 AI**：知识库收录 UniProt 和论文，Claude 带出处回答；Agent 自己调用 API 和知识库，护栏把关（11–12 号）
7. **运维**：Terraform 管全部 72 个 AWS 资源，GitHub Actions 每次提交跑 6 项检查，CloudWatch 告警（13 号）

### 面试怎么说
> "Sources land in S3 through a fetch-and-validate Lambda; Databricks turns them into bronze, silver and gold tables with dbt; models are trained with leakage-proof validation, tracked in MLflow and released through a gated Unity Catalog registry; 218 million variants are batch-scored and exported to an S3 lake queried by Athena behind a serverless API; a Bedrock knowledge base and a guarded Claude agent sit on top."

---

## 第 5 步 · 讲稿

### 在干什么
准备好三个长度的项目介绍，以及面试官最常追问的问题。

### 背景
面试官听项目介绍，最想知道四件事：**你解决了什么问题、你怎么设计的、结果怎么样（包括不好的）、你自己学到了什么**。所以讲稿按这个顺序。

**30 秒版**（电梯里、HR 电话里）：
> "I rebuilt my published variant-effect method, QAFI, as a production ML platform on AWS and Databricks. It scores all 218 million possible missense variants of the human proteome with two scores, stability and function, serves them through a serverless API, and adds a cited question-answering assistant and a guarded agent on top. Everything is in Terraform with CI/CD."

**2 分钟版**（"walk me through your project"）：
> "QAFI started as research code for predicting the effect of missense variants. I rebuilt it the way a company would.
>
> Data: public sources land in S3 through a Lambda that validates and registers every file. Databricks turns them into bronze, silver and gold tables; dbt owns the gold layer with tests.
>
> Models: features come from the ESM-2 protein language model, AlphaFold structures and physicochemical properties. I split proteins by sequence similarity, locked a test set and opened it once; every experiment is in MLflow. The final models are a small network for stability and an ensemble for function. On held-out proteins the stability score beat zero-shot ESM-2 in 96% of stability assays; on functional assays it ties AlphaMissense and CPT-1; on ClinVar it reaches an AUROC of 0.92 against 0.96 for AlphaMissense, so I'm clear it does not beat the best published predictor on clinical pathogenicity.
>
> Serving: 218 million scores are batch-computed, exported to S3 as sorted Parquet and queried with Athena; a lookup reads 6.5 megabytes of a 2.6-gigabyte table. A FastAPI container on Lambda behind API Gateway answers in about 0.15 seconds warm.
>
> GenAI: a Bedrock knowledge base over UniProt and my paper answers with citations, and an agent combines the API and the knowledge base, with guardrails that once caught the agent misreading two close numbers.
>
> Everything is Terraform with GitHub Actions; it costs a few dollars a month."

**常见追问**（每个先想一下自己怎么答，再看参考）：

| 追问 | 参考回答（要点） |
|---|---|
| Why batch scoring and not real-time inference? | The input space is enumerable (218M) and features are expensive; a lookup is cheaper, faster and reproducible. Real-time only for variants outside the reference proteome. |
| What would change at 100× traffic? | Athena's 1–3 s per query becomes the bottleneck: hot proteins into DynamoDB or a cache in front; the same image on ECS Fargate if cold starts matter. |
| How did you prevent leakage? | Proteins split by sequence similarity, not variants; the test set locked and opened once, enforced in code; features recomputed from raw with one SQL macro for training and serving, plus a parity test. |
| How do you know the model is still good? | Each release is gated on the held-out test, compared with the previous release, and validated on ClinVar; the API logs show what users ask. |
| What did not work? | Graph neural networks and LoRA fine-tuning added nothing over frozen ESM-2 embeddings; the bottleneck is functional training data, not architecture. |
| What would you do next? | More functional assays, hybrid (keyword + vector) retrieval for exact variant names, and a real-time path for non-reference sequences. |

### 公司里什么情况怎么做
- 讲项目时**主动说出不好的结果**（ClinVar 上输给 AlphaMissense、GNN 没用），面试官会更相信你其余的数字
- 每个数字都要能追溯：我们的每个数字都在 MLflow、notebook 输出或 CloudWatch 里

---

## 第 6 步 · 简历条目

### 在干什么
把这个项目写成简历上的几行。

### 背景
简历上的每一行：**动词开头，说做了什么，用了什么，结果是多少**。数字要真实、能解释。

**项目标题**：*QAFI ML Platform — proteome-scale variant effect prediction on AWS and Databricks* (github.com/pipi0616/qafi-ml-platform)

- Built an end-to-end ML platform that scores all **218 million** possible missense variants of the human proteome, from raw public data to a serverless API, defined in **Terraform (72 AWS resources)** with GitHub Actions CI/CD (6 checks, OIDC, no stored keys).
- Engineered a Databricks lakehouse (Unity Catalog, Delta, dbt with tests, Lakeflow Jobs triggered by Airflow) and S3 ingestion with Lambda validation for UniProt, ClinVar, ProteinGym, MaveDB and AlphaFold.
- Trained stability and function models on ESM-2 embeddings and AlphaFold features with similarity-grouped CV, Optuna tuning, MLflow tracking and a gated Unity Catalog registry; on a locked test set the stability score beat zero-shot ESM-2 in **96%** of stability assays, and the function score matched AlphaMissense on functional assays.
- Served the scores from an S3/Athena lake through a FastAPI container on Lambda behind IAM-authenticated API Gateway (**~0.15 s** warm, a lookup scans **6.5 MB** of 2.6 GB).
- Built a Bedrock RAG over 20,000 UniProt entries and the QAFI paper with cited answers, and a Claude tool-use agent with Bedrock Guardrails, evaluated on fixed exams (retrieval 11/11, agent 7/7).

**一句话自我介绍**（LinkedIn 标题之类）：*Bioinformatician turned ML engineer: I build data and ML platforms on AWS and Databricks, from lakehouse to GenAI.*

### 公司里什么情况怎么做
- 投不同的职位，换不同的重点：数据工程师强调 2、1 条；ML 工程师强调 3、4 条；AI/GenAI 工程师强调 5 条；云架构师强调 1、4 条
- README 是招聘方最先看到的：仓库首页已经按这一章更新

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 13 operations: alarms, dashboard, logs, costs; architecture and project story") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 全部完成

| 部分 | 章节 | 你会了什么 |
|---|---|---|
| A 基础 | 00–02 | 云账号安全、Terraform、工程化代码库和 CI/CD |
| B 数据平台 | 03–06 | S3 接入、Databricks 湖仓、dbt、编排 |
| C 机器学习 | 07–09 | 特征、防泄漏验证、深度学习、门禁、模型注册、全蛋白组打分 |
| D 服务和 AI | 10–12 | 数据湖和 Athena、无服务器 API、RAG、Agent 和护栏 |
| E 收尾 | 13 | 告警、日志、成本，以及怎么讲这个项目 |

**一句话**：从公开数据到一个能被提问的 AI 助手，每一步都按公司的做法搭起来、测过、能复现，并且你能讲清楚每一步为什么这样做、结果是多少、哪里还不够好。